<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/04-RAG_with_VectorStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG with ChromaDB Vector Store

This notebook demonstrates how to build a **Retrieval-Augmented Generation (RAG)** pipeline using [ChromaDB](https://www.trychroma.com/) as the vector store. It covers two popular framework integrations side by side:

- **Part 1 — LlamaIndex:** Store document chunks in ChromaDB, build a `VectorStoreIndex`, and query with Google Gemini.
- **Part 2 — LangChain:** Load the same chunks into a `Chroma` collection using `OpenAIEmbeddings`, then retrieve and answer with `ChatGoogleGenerativeAI` via `RetrievalQA`.

## Learning Objectives

By the end of this notebook you will be able to:
1. Chunk a raw text corpus and persist it in ChromaDB using both LlamaIndex and LangChain.
2. Build a query engine / retrieval chain that fetches relevant chunks and synthesises an answer with an LLM.
3. Compare how LlamaIndex and LangChain handle the same RAG workflow.

## Prerequisites

- Basic Python knowledge
- An **OpenAI API key** (used for embeddings)
- A **Google API key** with Gemini access (used for answer generation)
- Familiarity with vector embeddings and similarity search (helpful but not required)

## Install Packages and Setup Variables

In [ ]:
!pip install -q llama-index==0.14.19 openai==2.8.1 google-genai==1.70.0 llama-index-embeddings-openai==0.6.0 llama-index-llms-google-genai==0.9.0 \
                opentelemetry-api==1.38.0 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 \
                langchain==1.2.14 langchain-openai==1.1.8 langchain-core==1.2.25 langchain-chroma==1.1.0 langchain-google-genai==4.2.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.9/506.9 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/

In [ ]:
import os
# Set the following API keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
except ImportError:
    # Running locally — keys are expected in environment variables
    pass

import nest_asyncio
nest_asyncio.apply()

# Load the Dataset (CSV)

## Download

The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA 2 model. Read the dataset as a long string.

In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  169k  100  169k    0     0   851k      0 --:--:-- --:--:-- --:--:--  852k


## Read File

In [ ]:
import csv

text = ""

# Load the file as a CSV and concatenate all article text
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
        text += row[1]

# The number of characters in the dataset
print(len(text))

171044


# Chunking

In [ ]:
chunk_size = 512
chunks = []

# Split the long text into smaller, manageable chunks of 512 characters
for i in range(0, len(text), chunk_size):
    chunks.append(text[i : i + chunk_size])

print(len(chunks))

335


# Part 1: ChromaDB with LlamaIndex

In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them
documents = [Document(text=t) for t in chunks]

## Save on Chroma

In [ ]:
import chromadb

# Create a ChromaDB client and a new collection.
# PersistentClient saves data to disk at the specified path (creates the directory if it doesn't exist).
chroma_client = chromadb.PersistentClient(path="./mini-chunked-dataset")
chroma_collection = chroma_client.get_or_create_collection("mini-chunked-dataset")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# Define a storage context object using the created vector database
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding

# Build the index and generate embeddings using the OpenAI embedding model
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
    storage_context=storage_context,
    show_progress=True,
)

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/335 [00:00<?, ?it/s]

## Optional: Inspecting the ChromaDB Collection

In [ ]:
# Peek at the ChromaDB collection to verify documents were stored correctly
print(f"Collection name  : {chroma_collection.name}")
print(f"Total documents  : {chroma_collection.count()}")

# Preview first 3 documents
peek = chroma_collection.peek(limit=3)
print(f"\nFirst 3 document IDs : {peek['ids']}")
print(f"\nSample document text preview:")
for i, doc in enumerate(peek['documents']):
    print(f"  [{i+1}] {doc[:150]}...")

Collection name  : mini-chunked-dataset
Total documents  : 335

First 3 document IDs : ['717f8d27-618b-4b11-9759-817e45564ef4', '33e26ded-3e8d-44c5-9d6a-88cf1f217684', '2857dbfb-2f61-465d-8bf0-8eb1d4bc93ab']

Sample document text preview:
  [1] LLM Variants and Meta's Open Source Before shedding light on four major trends, I'd share the latest Meta's Llama 2 and Code Llama. Meta's Llama 2 rep...
  [2] c evaluations, focusing on safety and utility metrics, positioned Llama 2-Chat as a potential contender against proprietary, closed-source counterpart...
  [3] ational code model;Codel Llama - Python specialized for Python;and Code Llama - Instruct, which is fine-tuned for understanding natural language instr...


## Query Dataset

In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text
# and using an LLM to formulate the final answer

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-2.5-flash", max_tokens=512, temperature=1)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
response = query_engine.query("How many parameters does the LLaMA 2 model have?")
print(response)

Llama 2 is offered in four different model sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.


# Part 2: ChromaDB with LangChain

In [ ]:
from langchain_core.documents import Document

# Convert the chunks to Document objects so the LangChain framework can process them
documents = [Document(page_content=t) for t in chunks]

## Save on Chroma

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Add the documents to ChromaDB and create the index / embeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./mini-chunked-dataset",
    collection_name="mini-chunked-dataset",
)

## Query Dataset

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the LLM model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=512,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

query = "How many parameters does the LLaMA 2 model have?"

retriever = chroma_db.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert answering questions from retrieved content. "
     "Only answer AI-related questions, else say you cannot answer. "
     "Use the context below to answer concisely.\n\nContext: {context}"),
    ("human", "{input}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

response = rag_chain.invoke(query)
print(response)

LLaMA 2 models are available in four different sizes: 7 billion, 13 billion, 34 billion, and 70 billion parameters.


## Optional: LlamaIndex vs LangChain — Response Comparison

In [ ]:
comparison_query = "What safety measures does LLaMA 2 have?"
print(f"Query: {comparison_query}\n")
print("=" * 60)

# LlamaIndex response
try:
    li_response = query_engine.query(comparison_query)
    print("LlamaIndex response:")
    print(str(li_response)[:500])
except Exception as e:
    print(f"LlamaIndex error: {e}")

print("\n" + "-" * 60)

# LangChain response
try:
    lc_answer = rag_chain.invoke(comparison_query)
    print("LangChain response:")
    print(lc_answer[:500])
except Exception as e:
    print(f"LangChain error: {e}")

Query: What safety measures does LLaMA 2 have?

LlamaIndex response:
Llama 2-Chat underwent evaluations focusing on safety and utility metrics. The development of Llama 2 emphasized rigorous fine-tuning methodologies, and Meta's transparent description of these processes is intended to encourage community-driven advancements in large language models and promote collaborative and responsible AI development.

------------------------------------------------------------
LangChain response:
I cannot answer this question as the provided context does not contain information about the safety measures of LLaMA 2. It discusses LLaMA models generally, their parameters, training data, and access restrictions.
